# Review Sentiment Analyzer (Deep Learning)

This notebook classifies social-media reviews/comments as **negative**, **neutral** or **positive** using a
pre-trained **transformer** (RoBERTa fine-tuned on ~124M tweets), then analyses the results by platform and over time.

**Why this model?** Your data comes from Twitter, Instagram, Facebook and Reddit: short, informal text with emoji,
slang and ALL-CAPS emphasis. `cardiffnlp/twitter-roberta-base-sentiment-latest` was trained on exactly that kind of text
and returns three classes, so "meh" and "it's fine" can land in *neutral* instead of being forced into positive/negative.

**Pipeline**
1. Load & explore the data
2. Light text normalisation (we keep emoji and capitalisation, since the model uses them as signal)
3. Run the transformer to get class probabilities
4. Analyse: distribution, per-platform breakdown, trend over time, low-confidence cases
5. Sanity-check against hand labels
6. Try your own reviews interactively
7. *(Optional)* Fine-tune on your own labelled data

> `sample_data.csv` has **no sentiment labels**, so the main flow uses a pre-trained model (zero-shot with respect to your data).
> Section 7 shows how to fine-tune once you have labels.

## 1. Setup

In [ ]:
# Run once (remove the leading '#' if packages are missing)
# %pip install -q transformers torch pandas numpy matplotlib seaborn scikit-learn accelerate

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import pipeline

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

# ---- Configuration -------------------------------------------------------
DATA_PATH   = "sample_data.csv"      # change if your file lives elsewhere
TEXT_COL    = "comment"
MODEL_ID    = "cardiffnlp/twitter-roberta-base-sentiment-latest"
BATCH_SIZE  = 16
MAX_LEN     = 128                    # social posts are short; 128 tokens is plenty
LOW_CONF    = 0.60                   # below this, flag a prediction for human review
LABELS      = ["negative", "neutral", "positive"]
PALETTE     = {"negative": "#d9534f", "neutral": "#9aa0a6", "positive": "#2e9d5b"}

DEVICE = 0 if torch.cuda.is_available() else -1
print(f"Torch {torch.__version__} | running on {'GPU' if DEVICE == 0 else 'CPU'}")

## 2. Load and explore the data

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["date"])
print(f"{len(df)} rows, columns: {list(df.columns)}")
print("Missing values:\n", df.isna().sum().to_string())
print("Duplicate comments:", df[TEXT_COL].duplicated().sum())
df.head()

In [ ]:
df["n_words"] = df[TEXT_COL].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
df["platform"].value_counts().plot.bar(ax=axes[0], color="#4c78a8", rot=0)
axes[0].set_title("Comments per platform"); axes[0].set_xlabel(""); axes[0].set_ylabel("count")

sns.histplot(df["n_words"], bins=range(0, df["n_words"].max() + 2), ax=axes[1], color="#4c78a8")
axes[1].set_title("Comment length (words)"); axes[1].set_xlabel("words")
plt.tight_layout(); plt.show()

## 3. Text normalisation

Transformers need very little cleaning. We deliberately **keep** emoji, punctuation and capitalisation
("AGAIN", "!!") because they carry sentiment. We only replace user handles and URLs with placeholders, which is the
same preprocessing the model saw during training.

In [ ]:
def normalise(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"@\w+", "@user", text)            # handles -> @user
    text = re.sub(r"https?://\S+", "http", text)     # URLs    -> http
    text = re.sub(r"\s+", " ", text)                 # collapse whitespace
    return text

df["clean_text"] = df[TEXT_COL].map(normalise)
df[[TEXT_COL, "clean_text"]].head(3)

## 4. Load the model and predict

The first run downloads the weights (~500 MB) from the Hugging Face Hub; after that they are cached.
`top_k=None` makes the pipeline return the probability of **every** class, not just the winner, so we can measure confidence.

In [ ]:
clf = pipeline(
    "text-classification",
    model=MODEL_ID,
    tokenizer=MODEL_ID,
    top_k=None,
    device=DEVICE,
)
print("Model labels:", clf.model.config.id2label)

In [ ]:
# Some checkpoints expose generic names (LABEL_0/1/2); map them to readable ones.
GENERIC = {"LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive"}

def predict(texts, batch_size=BATCH_SIZE) -> pd.DataFrame:
    """Return a DataFrame of class probabilities + predicted label + confidence."""
    outputs = clf(list(texts), batch_size=batch_size, truncation=True, max_length=MAX_LEN)
    rows = []
    for scores in outputs:
        probs = {GENERIC.get(s["label"], s["label"]).lower(): s["score"] for s in scores}
        rows.append({lab: probs.get(lab, 0.0) for lab in LABELS})
    out = pd.DataFrame(rows)
    out["sentiment"] = out[LABELS].idxmax(axis=1)
    out["confidence"] = out[LABELS].max(axis=1)
    return out

preds = predict(df["clean_text"])
result = pd.concat([df.reset_index(drop=True), preds], axis=1)
result["net_score"] = result["positive"] - result["negative"]   # -1 (very negative) .. +1 (very positive)
result["needs_review"] = result["confidence"] < LOW_CONF
result[["platform", TEXT_COL, "sentiment", "confidence"]].round(3)

## 5. Analyse the results

### 5.1 Overall sentiment distribution

In [ ]:
counts = result["sentiment"].value_counts().reindex(LABELS, fill_value=0)
pct = (counts / counts.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(counts.index, counts.values, color=[PALETTE[l] for l in counts.index])
for b, p in zip(bars, pct):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.1, f"{p}%", ha="center")
ax.set_title("Overall sentiment"); ax.set_ylabel("comments")
plt.tight_layout(); plt.show()
print(f"Average net sentiment score: {result['net_score'].mean():+.2f}  (range -1 to +1)")

### 5.2 By platform

In [ ]:
by_platform = (
    result.groupby(["platform", "sentiment"]).size()
          .unstack(fill_value=0).reindex(columns=LABELS, fill_value=0)
)
share = by_platform.div(by_platform.sum(axis=1), axis=0) * 100

ax = share.plot.barh(stacked=True, figsize=(8, 3.5), color=[PALETTE[l] for l in LABELS])
ax.set_xlabel("% of comments"); ax.set_ylabel(""); ax.set_title("Sentiment mix by platform")
ax.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.show()

summary = result.groupby("platform").agg(
    comments=("sentiment", "size"),
    avg_net_score=("net_score", "mean"),
    avg_confidence=("confidence", "mean"),
).round(2).sort_values("avg_net_score")
summary

### 5.3 Trend over time
Daily average of the net sentiment score (positive probability minus negative probability).
With few comments per day the line will be noisy, so treat it as directional.

In [ ]:
daily = result.groupby("date").agg(net=("net_score", "mean"), n=("net_score", "size")).reset_index()

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.axhline(0, color="black", lw=0.8)
ax.plot(daily["date"], daily["net"], marker="o", color="#4c78a8")
ax.fill_between(daily["date"], daily["net"], 0, where=daily["net"] >= 0, color=PALETTE["positive"], alpha=0.2)
ax.fill_between(daily["date"], daily["net"], 0, where=daily["net"] < 0,  color=PALETTE["negative"], alpha=0.2)
ax.set_ylim(-1, 1); ax.set_title("Daily net sentiment"); ax.set_ylabel("net score")
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

### 5.4 Most negative comments (what to fix first)

In [ ]:
pd.set_option("display.max_colwidth", 120)
result.nsmallest(5, "net_score")[["platform", "date", TEXT_COL, "negative"]].round(3)

### 5.5 Low-confidence predictions (candidates for human review)

In [ ]:
review = result[result["needs_review"]].sort_values("confidence")
print(f"{len(review)} of {len(result)} predictions are below {LOW_CONF:.0%} confidence")
review[[TEXT_COL, "sentiment", "confidence"] + LABELS].round(2)

## 6. Sanity check against hand labels

`sample_data.csv` has no ground truth, so below are **quick manual labels for the 20 sample rows** (my own reading of each
comment; a couple, like *"Honestly not bad"*, are genuinely ambiguous). They are only meant to show whether the model is in
the right ballpark. If your data has a `label` column (values `negative`/`neutral`/`positive`) it is used instead.
With 20 rows the numbers are indicative, not a real benchmark.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

DEMO_LABELS = [
    "positive", "positive", "negative", "positive", "neutral",
    "positive", "negative", "positive", "negative", "positive",
    "neutral",  "negative", "positive", "negative", "negative",
    "positive", "neutral",  "negative", "positive", "neutral",
]

if "label" in result.columns:
    result["true_label"] = result["label"].str.lower()
elif len(result) == len(DEMO_LABELS):
    result["true_label"] = DEMO_LABELS
else:
    result["true_label"] = np.nan
    print("No labels available: skipping evaluation.")

if result["true_label"].notna().any():
    y_true, y_pred = result["true_label"], result["sentiment"]
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.2%}\n")
    print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    fig, ax = plt.subplots(figsize=(4.5, 3.8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=LABELS, yticklabels=LABELS, ax=ax)
    ax.set_xlabel("predicted"); ax.set_ylabel("actual"); ax.set_title("Confusion matrix")
    plt.tight_layout(); plt.show()

    wrong = result[result["true_label"] != result["sentiment"]]
    display(wrong[[TEXT_COL, "true_label", "sentiment", "confidence"]].round(2))

## 7. Try your own reviews

In [ ]:
def analyze(*reviews: str) -> pd.DataFrame:
    """Score one or more reviews and print a readable summary."""
    texts = [normalise(r) for r in reviews]
    out = predict(texts)
    out.insert(0, "review", reviews)
    for _, r in out.iterrows():
        print(f"{r['sentiment'].upper():9s} ({r['confidence']:.0%})  {r['review']}")
    return out.round(3)

analyze(
    "Absolutely love this app, best update yet 😍",
    "Support never answered my ticket. Unacceptable.",
    "It works. Nothing more, nothing less.",
    "Great, another crash. Just what I needed today.",       # sarcasm: a known weak spot
)

## 8. Export results

In [ ]:
export_cols = ["platform", "date", TEXT_COL, "sentiment", "confidence",
               "negative", "neutral", "positive", "net_score", "needs_review"]
result[export_cols].round(4).to_csv("sentiment_results.csv", index=False)
print("Saved sentiment_results.csv")

## 9. (Optional) Fine-tune on your own labelled data

A pre-trained model is a strong baseline, but it can be improved on domain-specific language (your product names, jargon,
sarcasm patterns). To do that you need **labelled** reviews, ideally a few hundred or more (a handful of rows will overfit).

1. Add a `label` column to your CSV with `negative`, `neutral` or `positive`.
2. Set `RUN_FINETUNE = True` and re-run the cell below.

The function trains with a stratified train/validation split, evaluates after each epoch, and saves the best model to
`./sentiment-finetuned`, which you can load by setting `MODEL_ID` to that path in Section 1.

In [ ]:
RUN_FINETUNE = False    # flip to True once your data has a `label` column and enough rows

def fine_tune(data: pd.DataFrame, text_col=TEXT_COL, label_col="label",
              base_model=MODEL_ID, out_dir="./sentiment-finetuned",
              epochs=3, lr=2e-5, batch_size=16):
    from sklearn.model_selection import train_test_split
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              TrainingArguments, Trainer)

    label2id = {l: i for i, l in enumerate(LABELS)}
    id2label = {i: l for l, i in label2id.items()}

    data = data[[text_col, label_col]].dropna().copy()
    data["y"] = data[label_col].str.lower().map(label2id)
    data["x"] = data[text_col].map(normalise)
    train_df, val_df = train_test_split(data, test_size=0.2, stratify=data["y"], random_state=42)

    tok = AutoTokenizer.from_pretrained(base_model)
    model = AutoModelForSequenceClassification.from_pretrained(
        base_model, num_labels=3, id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True)

    class DS(torch.utils.data.Dataset):
        def __init__(self, d):
            self.enc = tok(list(d["x"]), truncation=True, max_length=MAX_LEN, padding=True)
            self.y = list(d["y"])
        def __len__(self): return len(self.y)
        def __getitem__(self, i):
            item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
            item["labels"] = torch.tensor(self.y[i])
            return item

    def metrics(p):
        from sklearn.metrics import accuracy_score, f1_score
        preds = p.predictions.argmax(-1)
        return {"accuracy": accuracy_score(p.label_ids, preds),
                "macro_f1": f1_score(p.label_ids, preds, average="macro")}

    args = TrainingArguments(
        output_dir=out_dir, num_train_epochs=epochs, learning_rate=lr,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=batch_size,
        eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model="macro_f1", weight_decay=0.01, report_to="none", logging_steps=10)

    trainer = Trainer(model=model, args=args, train_dataset=DS(train_df),
                      eval_dataset=DS(val_df), compute_metrics=metrics)
    trainer.train()
    print(trainer.evaluate())
    trainer.save_model(out_dir); tok.save_pretrained(out_dir)
    return trainer

if RUN_FINETUNE:
    assert "label" in df.columns, "Add a 'label' column (negative/neutral/positive) to your CSV first."
    trainer = fine_tune(df)
else:
    print("Fine-tuning skipped (RUN_FINETUNE = False).")